## Cell 0 — Drive Setup

In [1]:
# Cell 0 — Drive Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, time, json, glob, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOMSTATE = 42
np.random.seed(RANDOMSTATE)

# Unified Drive paths
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'iforeststrict'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('CACHEDIR     :', CACHEDIR)


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions
CACHEDIR     : /content/drive/MyDrive/tsad_ensemble_runs/_cache


## Cell 1 — Dataset Paths

In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'

CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

# Download NAB labels
NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'

if not os.path.exists(NABLABELSLOCAL):
    print('Downloading NAB labels...')
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)

with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

# Validation
assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found. Check Drive.'

nab_csvs = glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True)
nab_csvs = [f for f in nab_csvs if 'README' not in f]

print(f'CREDITCARDPATH : {CREDITCARDPATH}')
print(f'NABROOT        : {NABROOT}')
print(f'NAB CSV count  : {len(nab_csvs)}')
print(f'NAB labels     : {len(NABWINDOWSMAP)} entries, {sum(1 for v in NABWINDOWSMAP.values() if v)} with anomalies')


CREDITCARDPATH : /content/drive/MyDrive/creditcard.csv
NABROOT        : /content/drive/MyDrive/NAB Dataset
NAB CSV count  : 58
NAB labels     : 58 entries, 52 with anomalies


## Cell 2 — Shared Utilities

In [3]:
# Cell 2 — Shared Utilities

def robust_zscores(scores, eps=1e-6, clip=50.0):
    s = np.asarray(scores, dtype=float)
    med = np.median(s)
    mad = np.median(np.abs(s - med))
    return np.clip((s - med) / max(1.4826 * mad, eps), -clip, clip)

def keep_runs(y_pred, minlen=3):
    y = np.asarray(y_pred, dtype=int).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if j - i < minlen:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    # PA protocol: if any point in a contiguous anomaly segment is detected,
    # credit the entire segment as detected
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_pred, dtype=int).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def temporal_slices(n, valfrac=0.30):
    n = int(n)
    v = int(max(1, round(n * valfrac)))
    v = min(v, n - 1)
    return slice(0, v), slice(v, n)

def best_strict_f1_threshold(y, s, ngrid=300, qlo=0.50, qhi=0.999, minrun=3):
    y = np.asarray(y, dtype=int)
    s = np.asarray(s, dtype=float)
    if y.sum() == 0:
        return float(np.max(s)) if len(s) else 0.0, 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(s, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = keep_runs((s >= t).astype(int), minlen=minrun)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


## Cell 3 — Credit Card IF Agent

Improvements:
- RobustScaler instead of StandardScaler (better for skewed fraud data)
- More engineered features (V-feature interactions, amount quantiles)
- Grid search over contamination values on the tune set
- PR-curve based threshold selection on tune set
- Both strict and PA metrics reported
- Entity ID = `creditcard` for coordinator compatibility


In [4]:
# Cell 3 — Credit Card IF Agent

class CreditCardIFAgent:
    def __init__(self):
        self.model = None
        self.scaler = RobustScaler()
        self.threshold = None
        self.features = None

    def engineer_features(self, df):
        df = df.copy()
        # Time-based features
        df['hoursin'] = np.sin(2 * np.pi * df['Time'] / 86400.0)
        df['hourcos'] = np.cos(2 * np.pi * df['Time'] / 86400.0)
        # Amount features
        df['Amountlog'] = np.log1p(df['Amount'])
        df['AmountSq'] = df['Amount'] ** 2
        # V-feature interactions (top discriminative pairs from literature)
        df['V1xV2'] = df['V1'] * df['V2']
        df['V3xV4'] = df['V3'] * df['V4']
        df['V14xV17'] = df['V14'] * df['V17']
        df['V12xV14'] = df['V12'] * df['V14']
        # Absolute values of key V features (fraud tends to have extreme values)
        for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
            df[f'{v}_abs'] = df[v].abs()

        vcols = [f'V{i}' for i in range(1, 29)]
        extra = ['Amountlog', 'AmountSq', 'hoursin', 'hourcos',
                 'V1xV2', 'V3xV4', 'V14xV17', 'V12xV14',
                 'V1_abs', 'V3_abs', 'V4_abs', 'V7_abs',
                 'V10_abs', 'V12_abs', 'V14_abs', 'V17_abs']
        self.features = vcols + extra
        return df[self.features]

    def load(self):
        df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
        y = df['Class'].astype(int).values
        X = self.engineer_features(df)
        return X, y

    def fit(self):
        Xall, yall = self.load()

        # Split: 80% tune, 20% test (same as Memto for alignment)
        idx = np.arange(len(yall), dtype=np.int64)
        idxtune, self.idxtest = train_test_split(
            idx, test_size=0.20, stratify=yall, random_state=RANDOMSTATE
        )

        # Fit scaler on normal samples from tune set
        Xtune = Xall.iloc[idxtune]
        ytune = yall[idxtune]
        Xnormal_tune = Xtune[ytune == 0]
        self.scaler.fit(Xnormal_tune)

        # Grid search contamination on tune set
        contamination_rates = [0.001, 0.002, 0.005, 0.01]
        best_cont, best_f1_tune = 0.001, -1.0

        for cont in contamination_rates:
            model = IsolationForest(
                n_estimators=400,
                contamination=cont,
                max_samples=min(50000, len(Xnormal_tune)),
                max_features=0.8,
                n_jobs=-1,
                random_state=RANDOMSTATE,
            )
            model.fit(self.scaler.transform(Xnormal_tune))

            scores_tune = -model.decision_function(self.scaler.transform(Xtune))
            # PR-curve threshold on tune set
            prec, rec, thresholds = precision_recall_curve(ytune, scores_tune)
            f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
            if len(f1s) > 0:
                best_idx = int(np.argmax(f1s))
                f1v = float(f1s[best_idx])
                if f1v > best_f1_tune:
                    best_f1_tune = f1v
                    best_cont = cont
                    self.model = model
                    self.threshold = float(thresholds[best_idx])

        print(f'Best contamination: {best_cont}, tune F1: {best_f1_tune:.4f}, threshold: {self.threshold:.6f}')
        return self

    def evaluate(self):
        Xall, yall = self.load()
        idxtest = self.idxtest

        Xtest = Xall.iloc[idxtest]
        ytest = yall[idxtest].astype(np.int8)
        scores = (-self.model.decision_function(self.scaler.transform(Xtest))).astype(np.float32)
        preds = (scores >= self.threshold).astype(np.int8)

        # Strict metrics
        strict = compute_metrics(ytest, preds, scores, prefix='')

        results = {
            'dataset': 'creditcard',
            'protocol': 'strict point-wise holdout (20% test)',
            **strict,
            'threshold': float(self.threshold),
            'n_anomalies_true': int(ytest.sum()),
            'n_anomalies_pred': int(preds.sum()),
        }

        return results, scores, ytest, preds, idxtest.astype(np.int64)

    def save(self):
        joblib.dump({
            'model': self.model,
            'scaler': self.scaler,
            'features': self.features,
            'threshold': self.threshold,
            'test_indices': self.idxtest,
        }, os.path.join(ARTIFACTDIR, 'ifcreditcard.joblib'))
        print('Saved CC artifact.')

ccagent = CreditCardIFAgent()
print('Fitting Credit Card IF...')
ccagent.fit()

print('Evaluating...')
ccresults, ccscores, ccytrue, ccpred, cc_original_row_id = ccagent.evaluate()
ccagent.save()

print()
print('=' * 60)
print('CREDIT CARD IF RESULTS')
print('=' * 60)
for k, v in ccresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


Fitting Credit Card IF...
Best contamination: 0.001, tune F1: 0.5161, threshold: -0.008910
Evaluating...
Saved CC artifact.

CREDIT CARD IF RESULTS
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.464000
  recall                   : 0.591837
  f1                       : 0.520179
  rocauc                   : 0.959610
  prauc                    : 0.506539
  threshold                : -0.008910
  n_anomalies_true         : 98
  n_anomalies_pred         : 125


## Cell 4 — NAB IF Agent

NAB contains 58 CSV files organized in 7 categories:
- `realAWSCloudwatch/` — AWS server metrics
- `realAdExchange/` — ad click rates
- `realKnownCause/` — data with documented anomaly causes
- `realTraffic/` — web traffic metrics
- `realTweets/` — Twitter volume data
- `artificialNoAnomaly/` — synthetic normal data
- `artificialWithAnomaly/` — synthetic data with injected anomalies

Key improvements:
- Label matching uses relative paths (subfolder/filename) to match NAB's `combined_windows.json` keys
- Richer feature engineering (more rolling windows, rate of change, entropy proxy)
- Trains on first 40% normal-only data, tunes threshold on next 30%, evaluates on final 30%
- Reports both strict AND point-adjusted metrics
- Per-category breakdown


In [5]:
# Cell 4 — NAB IF Agent

class NABIFAgent:
    WINDOW_SIZES = [10, 30, 60]   # multiple scales
    MINRUN = 3
    THRGRID = 300
    TRAINFRAC = 0.40
    VALFRAC = 0.30                 # val is TRAINFRAC..TRAINFRAC+VALFRAC
    CONTAMINATION = 0.005

    def __init__(self):
        self.series_models = {}

    def match_nab_key(self, csv_path):
        # NAB labels use keys like 'realAWSCloudwatch/ec2_cpu_utilization.csv'
        # We need to find the relative path from NABROOT that matches
        rel = os.path.relpath(csv_path, NABROOT).replace('\\', '/')
        if rel in NABWINDOWSMAP:
            return rel
        # Fallback: try matching by basename
        base = os.path.basename(csv_path)
        for k in NABWINDOWSMAP:
            if os.path.basename(k) == base:
                return k
        return None

    def make_labels(self, timestamps, csv_path):
        y = np.zeros(len(timestamps), dtype=np.int8)
        key = self.match_nab_key(csv_path)
        if key is None or not NABWINDOWSMAP.get(key):
            return y
        for t0, t1 in NABWINDOWSMAP[key]:
            start = pd.Timestamp(t0)
            end = pd.Timestamp(t1)
            y[(timestamps >= start) & (timestamps <= end)] = 1
        return y

    def extract_features(self, values):
        s = pd.Series(values.astype(np.float64))
        parts = [s.rename('value')]
        for w in self.WINDOW_SIZES:
            roll = s.rolling(w, min_periods=1)
            parts.extend([
                roll.mean().rename(f'rmean_{w}'),
                roll.std().fillna(0).rename(f'rstd_{w}'),
                roll.min().rename(f'rmin_{w}'),
                roll.max().rename(f'rmax_{w}'),
                (roll.max() - roll.min()).rename(f'rrange_{w}'),
            ])
        # Differences at multiple scales
        parts.extend([
            s.diff(1).fillna(0).rename('diff1'),
            s.diff(3).fillna(0).rename('diff3'),
            s.diff(10).fillna(0).rename('diff10'),
        ])
        # Z-score relative to rolling window
        rmean = s.rolling(30, min_periods=1).mean()
        rstd = s.rolling(30, min_periods=1).std().fillna(1) + 1e-9
        parts.append(((s - rmean) / rstd).rename('zscore_30'))
        # Rate of change
        parts.append(s.pct_change().fillna(0).clip(-10, 10).rename('pct_change'))

        df = pd.concat(parts, axis=1)
        return np.nan_to_num(df.values, nan=0.0, posinf=0.0, neginf=0.0)

    def load_csv(self, path):
        df = pd.read_csv(path, parse_dates=['timestamp'])
        return df['value'].values, df['timestamp'].values

    def fit(self):
        all_csv = sorted(glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True))
        all_csv = [f for f in all_csv if 'README' not in f]
        trained = 0
        skipped = 0

        for path in all_csv:
            try:
                vals, ts = self.load_csv(path)
                y = self.make_labels(ts, path)
                X = self.extract_features(vals)

                n = len(y)
                n_tr = max(50, int(round(n * self.TRAINFRAC)))
                tr_idx = np.arange(0, min(n_tr, n))

                # Train only on normal data
                if y[tr_idx].sum() > 0:
                    tr_mask = y[tr_idx] == 0
                    X_tr = X[tr_idx][tr_mask]
                else:
                    X_tr = X[tr_idx]

                if len(X_tr) < 30:
                    skipped += 1
                    continue

                scaler = StandardScaler().fit(X_tr)
                model = IsolationForest(
                    n_estimators=300,
                    contamination=self.CONTAMINATION,
                    max_samples=min(5000, len(X_tr)),
                    max_features=0.8,
                    n_jobs=-1,
                    random_state=RANDOMSTATE,
                )
                model.fit(scaler.transform(X_tr))

                # Use relative path as key for coordinator compatibility
                rel_key = os.path.relpath(path, NABROOT).replace('\\', '/')
                self.series_models[rel_key] = {
                    'model': model,
                    'scaler': scaler,
                    'csv_path': path,
                }
                trained += 1

            except Exception as e:
                print(f'[NAB-IF] fit skip {os.path.basename(path)}: {e}')
                skipped += 1

        print(f'[NAB-IF] Trained {trained} series, skipped {skipped}')
        return self

    def evaluate(self):
        bundle = {}
        S_all, Y_all, P_all, PA_all = [], [], [], []
        category_results = {}

        for rel_key, mm in self.series_models.items():
            try:
                vals, ts = self.load_csv(mm['csv_path'])
                y = self.make_labels(ts, mm['csv_path'])
                X = self.extract_features(vals)

                scores = -mm['model'].decision_function(mm['scaler'].transform(X))
                s = robust_zscores(scores)

                n = len(y)
                n_tr = int(round(n * self.TRAINFRAC))
                n_val = int(round(n * self.VALFRAC))
                sl_val = slice(n_tr, n_tr + n_val)
                sl_test = slice(n_tr + n_val, n)

                if sl_test.stop <= sl_test.start:
                    continue

                # Tune threshold on validation portion
                t, f_val = best_strict_f1_threshold(
                    y[sl_val], s[sl_val], ngrid=self.THRGRID, minrun=self.MINRUN
                )

                p_full = keep_runs((s >= t).astype(int), minlen=self.MINRUN)

                y_h = y[sl_test]
                s_h = s[sl_test]
                p_h = p_full[sl_test]
                pa_h = point_adjust(y_h, p_h)

                # Use basename as entity ID for coordinator compatibility
                entity_name = os.path.basename(rel_key)

                bundle[entity_name] = {
                    'entity_id': entity_name,
                    'nab_key': rel_key,
                    'scores_full': s.astype(np.float32),
                    'y_full': y.astype(np.int8),
                    'pred_full': p_full.astype(np.int8),
                    'row_id': np.arange(len(y), dtype=np.int64),
                    'val_slice': (sl_val.start, sl_val.stop),
                    'test_slice': (sl_test.start, sl_test.stop),
                    'threshold': float(t),
                    'val_f1': float(f_val),
                }

                S_all.append(s_h)
                Y_all.append(y_h)
                P_all.append(p_h)
                PA_all.append(pa_h)

                # Track per-category
                cat = rel_key.split('/')[0] if '/' in rel_key else 'unknown'
                if cat not in category_results:
                    category_results[cat] = {'S': [], 'Y': [], 'P': [], 'PA': []}
                category_results[cat]['S'].append(s_h)
                category_results[cat]['Y'].append(y_h)
                category_results[cat]['P'].append(p_h)
                category_results[cat]['PA'].append(pa_h)

            except Exception as e:
                print(f'[NAB-IF] eval skip {rel_key}: {e}')

        if not S_all:
            return {}, bundle, category_results

        S = np.concatenate(S_all)
        Y = np.concatenate(Y_all)
        P = np.concatenate(P_all)
        PA = np.concatenate(PA_all)

        strict = compute_metrics(Y, P, S, prefix='strict_')
        pa_metrics = compute_metrics(Y, PA, S, prefix='pa_')

        results = {
            'dataset': 'nab',
            'protocol': f'temporal holdout: train {int(100*self.TRAINFRAC)}% / val {int(100*self.VALFRAC)}% / test {int(100*(1-self.TRAINFRAC-self.VALFRAC))}%',
            **strict,
            **pa_metrics,
            'n_anomalies_true': int(Y.sum()),
            'n_anomalies_pred': int(P.sum()),
            'n_series': len(bundle),
        }

        return results, bundle, category_results

    def save(self, bundle):
        joblib.dump(bundle, os.path.join(PREDICTIONSDIR, 'ifnabstrictpointwise.joblib'))
        # Don't save the full model artifacts (huge); just predictions
        print('[NAB-IF] Saved prediction bundle.')

nabagent = NABIFAgent()
print('Fitting NAB IF...')
nabagent.fit()

print('Evaluating...')
nabresults, nabbundle, nab_cat_results = nabagent.evaluate()
nabagent.save(nabbundle)

print()
print('=' * 60)
print('NAB IF RESULTS (GLOBAL)')
print('=' * 60)
for k, v in nabresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


Fitting NAB IF...
[NAB-IF] Trained 58 series, skipped 0
Evaluating...
[NAB-IF] Saved prediction bundle.

NAB IF RESULTS (GLOBAL)
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.144322
  strict_recall            : 0.294312
  strict_f1                : 0.193672
  strict_rocauc            : 0.666530
  strict_prauc             : 0.215317
  pa_precision             : 0.318174
  pa_recall                : 0.814288
  pa_f1                    : 0.457561
  pa_rocauc                : 0.666530
  pa_prauc                 : 0.215317
  n_anomalies_true         : 11814
  n_anomalies_pred         : 24092
  n_series                 : 58


## Cell 5 — NAB Per-Category Breakdown

In [6]:
# Cell 5 — NAB Per-Category Breakdown

print('NAB PER-CATEGORY RESULTS')
print('=' * 90)

cat_rows = []
for cat, data in sorted(nab_cat_results.items()):
    Y = np.concatenate(data['Y'])
    P = np.concatenate(data['P'])
    S = np.concatenate(data['S'])
    PA = np.concatenate(data['PA'])

    strict = compute_metrics(Y, P, S)
    pa = compute_metrics(Y, PA, S, prefix='pa_')

    row = {
        'category': cat,
        'n_series': len(data['Y']),
        'n_points': len(Y),
        'n_anomalies': int(Y.sum()),
        'anomaly_rate': float(Y.mean()),
        **strict,
        **pa,
    }
    cat_rows.append(row)

    print(f"  {cat:30s}  series={row['n_series']:3d}  "
          f"strict F1={strict['f1']:.4f}  PA F1={pa['pa_f1']:.4f}  "
          f"anomaly_rate={row['anomaly_rate']:.4f}")

cat_df = pd.DataFrame(cat_rows)
display(cat_df)


NAB PER-CATEGORY RESULTS
  artificialNoAnomaly             series=  5  strict F1=0.0000  PA F1=0.0000  anomaly_rate=0.0000
  artificialWithAnomaly           series=  6  strict F1=0.3511  PA F1=0.5249  anomaly_rate=0.2308
  realAWSCloudwatch               series= 17  strict F1=0.1392  PA F1=0.2358  anomaly_rate=0.0943
  realAdExchange                  series=  6  strict F1=0.2755  PA F1=0.3486  anomaly_rate=0.0898
  realKnownCause                  series=  7  strict F1=0.1947  PA F1=0.6770  anomaly_rate=0.1873
  realTraffic                     series=  7  strict F1=0.2818  PA F1=0.6934  anomaly_rate=0.2160
  realTweets                      series= 10  strict F1=0.1694  PA F1=0.4428  anomaly_rate=0.0639


,category,n_series,n_points,n_anomalies,anomaly_rate,precision,recall,f1,rocauc,prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc
0,artificialNoAnomaly,5,6045,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN
1,artificialWithAnomaly,6,7254,1674,0.230769,0.281194,0.467145,0.351066,0.624590,0.470948,0.395342,0.780765,0.524900,0.624590,0.470948
2,realAWSCloudwatch,17,20315,1916,0.094315,0.087227,0.343946,0.139162,0.600334,0.234006,0.145900,0.614823,0.235836,0.600334,0.234006
3,realAdExchange,6,2884,259,0.089806,0.168385,0.756757,0.275474,0.679409,0.217261,0.211084,1.000000,0.348587,0.679409,0.217261
4,realKnownCause,7,20868,3908,0.187272,0.211729,0.180143,0.194663,0.784874,0.370568,0.560382,0.854913,0.677001,0.784874,0.370568
5,realTraffic,7,4699,1015,0.216003,0.293490,0.270936,0.281762,0.743346,0.551803,0.573454,0.876847,0.693416,0.743346,0.551803
6,realTweets,10,47590,3042,0.063921,0.120910,0.283037,0.169438,0.553408,0.113732,0.297024,0.869494,0.442789,0.553408,0.113732


## Cell 6 — Export for Coordinator

In [7]:
# Cell 6 — Export for Coordinator

exported = {}
summaryrows = []

# --- Credit Card ---
ccbundle = {
    'dataset': 'creditcard',
    'model': 'isolationforest',
    'protocol': 'strict point-wise holdout',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': ccscores,
            'yfull': ccytrue,
            'predfull': ccpred,
            'rowid': np.arange(len(ccytrue), dtype=np.int64),
            'originalrowid': cc_original_row_id,
            'threshold': float(ccagent.threshold),
        }
    }
}

ccpath = os.path.join(PREDICTIONSDIR, 'ifcreditcardstrictpointwise.joblib')
joblib.dump(ccbundle, ccpath)
exported['creditcard'] = ccpath
summaryrows.append({
    'dataset': 'creditcard',
    **{k: v for k, v in ccresults.items() if k != 'dataset'},
})
print('Saved', ccpath)

# --- NAB ---
nabpayload = {
    'dataset': 'nab',
    'model': 'isolationforest',
    'protocol': nabresults.get('protocol', 'temporal holdout'),
    'entities': nabbundle,
}
nabpath = os.path.join(PREDICTIONSDIR, 'ifnabstrictpointwise.joblib')
joblib.dump(nabpayload, nabpath)
exported['nab'] = nabpath
summaryrows.append({
    'dataset': 'nab',
    **{k: v for k, v in nabresults.items() if k != 'dataset'},
})
print('Saved', nabpath)

# --- Summary CSV ---
summary = pd.DataFrame(summaryrows)
summarypath = os.path.join(PREDICTIONSDIR, 'ifstrictsummary.csv')
summary.to_csv(summarypath, index=False)
print('Saved', summarypath)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID,
    'driveroot': DRIVEROOT,
    'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'isolationforest',
    'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR,
    'predictionsdir': PREDICTIONSDIR,
    'exports': exported,
    'summarycsv': summarypath,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard', 'nab'],
    'smd_dropped': True,
}
manifestpath = os.path.join(PREDICTIONSDIR, 'ifmanifest.json')
with open(manifestpath, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifestpath)

print()
print('Coordinator-ready files are in:', PREDICTIONSDIR)
print('Exported datasets:', sorted(exported.keys()))


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifcreditcardstrictpointwise.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifnabstrictpointwise.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifstrictsummary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred,...,strict_recall,strict_f1,strict_rocauc,strict_prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc,n_series
0,creditcard,strict point-wise holdout (20% test),0.464,0.591837,0.520179,0.95961,0.506539,-0.00891,98,125,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,nab,temporal holdout: train 40% / val 30% / test 30%,NaN,NaN,NaN,NaN,NaN,NaN,11814,24092,...,0.294312,0.193672,0.66653,0.215317,0.318174,0.814288,0.457561,0.66653,0.215317,58.0


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifmanifest.json

Coordinator-ready files are in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions
Exported datasets: ['creditcard', 'nab']


## Cell 7 — Final Summary

In [8]:
# Cell 7 — Final Summary

print('=' * 70)
print('ISOLATION FOREST FINAL SUMMARY')
print('=' * 70)

print()
print('CREDIT CARD:')
for k, v in ccresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB (global):')
for k, v in nabresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB (per-category):')
display(cat_df[['category', 'n_series', 'n_anomalies', 'f1', 'pa_f1', 'rocauc']].round(4))

print()
print('All exports saved to:', PREDICTIONSDIR)
print('Done.')


ISOLATION FOREST FINAL SUMMARY

CREDIT CARD:
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.464000
  recall                   : 0.591837
  f1                       : 0.520179
  rocauc                   : 0.959610
  prauc                    : 0.506539
  threshold                : -0.008910
  n_anomalies_true         : 98
  n_anomalies_pred         : 125

NAB (global):
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.144322
  strict_recall            : 0.294312
  strict_f1                : 0.193672
  strict_rocauc            : 0.666530
  strict_prauc             : 0.215317
  pa_precision             : 0.318174
  pa_recall                : 0.814288
  pa_f1                    : 0.457561
  pa_rocauc                : 0.666530
  pa_prauc                 : 0.215317
  n_anomalies_true         : 11814
  n_anomalies_

,category,n_series,n_anomalies,f1,pa_f1,rocauc
0,artificialNoAnomaly,5,0,0.0000,0.0000,NaN
1,artificialWithAnomaly,6,1674,0.3511,0.5249,0.6246
2,realAWSCloudwatch,17,1916,0.1392,0.2358,0.6003
3,realAdExchange,6,259,0.2755,0.3486,0.6794
4,realKnownCause,7,3908,0.1947,0.6770,0.7849
5,realTraffic,7,1015,0.2818,0.6934,0.7433
6,realTweets,10,3042,0.1694,0.4428,0.5534



All exports saved to: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions
Done.


In [9]:
# DEBUG — Isolation Forest contribution analysis
print('=== ISOLATION FOREST CONTRIBUTION ANALYSIS ===')
print()
print('IF ROLE IN ENSEMBLE:')
print('  - CC F1=0.52: moderate — catches ~59% of frauds')
print('  - High ROC-AUC (0.96): good anomaly ranking, threshold-sensitive')
print('  - NAB: provides tree-based anomaly perspective for time series')
print()
print('COMPLEMENTARITY with other models:')
print('  - IF uses random feature subspaces (different from ECOD\'s marginal approach)')
print('  - IF captures feature interactions via random partitioning')
print('  - Adds diversity to NAB ensemble (4 NAB models: Conv-AE, ECOD, IF, MP)')
print()
print('In the coordinator, IF gets low CC weight via F1^2 weighting:')
print(f'  Weight factor: {0.52**2:.4f} (vs XGBoost {0.86**2:.4f})')
print('  So IF has minimal influence on CC but adds NAB diversity.')


=== ISOLATION FOREST CONTRIBUTION ANALYSIS ===

IF ROLE IN ENSEMBLE:
  - CC F1=0.52: moderate — catches ~59% of frauds
  - High ROC-AUC (0.96): good anomaly ranking, threshold-sensitive
  - NAB: provides tree-based anomaly perspective for time series

COMPLEMENTARITY with other models:
  - IF uses random feature subspaces (different from ECOD's marginal approach)
  - IF captures feature interactions via random partitioning
  - Adds diversity to NAB ensemble (4 NAB models: Conv-AE, ECOD, IF, MP)

In the coordinator, IF gets low CC weight via F1^2 weighting:
  Weight factor: 0.2704 (vs XGBoost 0.7396)
  So IF has minimal influence on CC but adds NAB diversity.
